# Organic Session Decline Diagnostic — April 2026

**Goal:** Identify what is driving organic (SEO) sessions down in April 2026.

**Framework:** Organic sessions are a function of GSC impressions and click-through rate:

> Sessions ≈ Clicks = Impressions × CTR

If sessions are down, it must be one or both of:
1. **Impressions down** → we're ranking for fewer queries or in worse positions (fewer chances to show up)
2. **CTR down** → we're getting fewer clicks per impression (worse positions, SERP feature changes, or competitor snippets)

**Comparison periods:**
- **Current:** April 1-10, 2026 (GSC data available)
- **Prior:** March 1-10, 2026 (same day count for fair comparison)
- **YoY:** April 1-10, 2025

**Pacing:** DOW-weighted projection from Feb-Mar historical patterns to project full-month from partial data.

In [1]:
%sql
-- ====================================================================
-- STEP 0: Configuration — set comparison periods
-- ====================================================================
-- April MTD: limited to GSC data availability (lags 2-3 days)
-- Adjust curr_end if more GSC data becomes available.

CREATE OR REPLACE TEMP VIEW _config AS
SELECT
  DATE '2026-04-01'  AS curr_start,
  DATE '2026-04-10'  AS curr_end,
  DATE '2026-03-01'  AS prior_start,
  DATE '2026-03-10'  AS prior_end,
  DATE '2025-04-01'  AS yoy_start,
  DATE '2025-04-10'  AS yoy_end,
  30                 AS april_total_days;

SyntaxError: invalid character '—' (U+2014) (655074499.py, line 3)

In [ ]:
%sql
-- ====================================================================
-- STEP 1: DOW pacing weights from Feb-Mar 2026 sessions
-- ====================================================================
-- Used to project April full-month from partial data,
-- accounting for Mon-Fri vs weekend traffic patterns.

CREATE OR REPLACE TEMP VIEW dow_session_weights AS
WITH daily AS (
  SELECT _date, DAYOFWEEK(_date) AS dow, SUM(sessions) AS daily_sessions
  FROM energy_prod.data_science.mp_session_level_query
  WHERE _date >= '2026-02-01' AND _date < '2026-04-01'
    AND marketing_channel = 'Organic'
  GROUP BY _date, DAYOFWEEK(_date)
)
SELECT dow, AVG(daily_sessions) AS avg_daily_sessions
FROM daily
GROUP BY dow;

-- Calculate April pacing multiplier
SELECT
  ROUND(SUM(CASE WHEN a._date <= (SELECT curr_end FROM _config) THEN w.avg_daily_sessions END), 0) AS dow_weighted_mtd,
  ROUND(SUM(w.avg_daily_sessions), 0)  AS dow_weighted_full_month,
  ROUND(
    SUM(CASE WHEN a._date <= (SELECT curr_end FROM _config) THEN w.avg_daily_sessions END)
    / SUM(w.avg_daily_sessions) * 100, 1
  ) AS pct_of_month_elapsed,
  ROUND(
    SUM(w.avg_daily_sessions)
    / SUM(CASE WHEN a._date <= (SELECT curr_end FROM _config) THEN w.avg_daily_sessions END),
    3
  ) AS pacing_multiplier
FROM (
  SELECT EXPLODE(SEQUENCE(DATE '2026-04-01', DATE '2026-04-30')) AS _date
) a
JOIN dow_session_weights w ON DAYOFWEEK(a._date) = w.dow;

In [ ]:
%sql
-- ====================================================================
-- STEP 2: Daily GSC metrics for our domains
-- ====================================================================
-- Pulls daily clicks, impressions, CTR, and avg position
-- for March and April comparison.

CREATE OR REPLACE TEMP VIEW gsc_daily AS
SELECT
  date,
  SUM(clicks)       AS clicks,
  SUM(impressions)  AS impressions,
  ROUND(SUM(clicks) * 100.0 / NULLIF(SUM(impressions), 0), 3) AS ctr_pct,
  ROUND(
    SUM(position * impressions) / NULLIF(SUM(impressions), 0), 1
  ) AS avg_position
FROM lakehouse_production.common.gsc_search_analytics_d_1
WHERE date >= '2025-03-01'
  AND domain IN (
    'choosetexaspower.org','saveonenergy.com',
    'chooseenergy.com','texaselectricrates.com'
  )
GROUP BY date
ORDER BY date;

In [ ]:
%sql
-- ====================================================================
-- STEP 3: Daily organic sessions
-- ====================================================================

CREATE OR REPLACE TEMP VIEW sessions_daily AS
SELECT
  _date,
  SUM(sessions) AS sessions,
  SUM(cart_orders) + SUM(phone_orders) AS orders,
  SUM(zip_entry)     AS zip_entries,
  SUM(has_cart)       AS carts
FROM energy_prod.data_science.mp_session_level_query
WHERE _date >= '2025-03-01'
  AND marketing_channel = 'Organic'
GROUP BY _date
ORDER BY _date;

In [ ]:
%sql
-- ====================================================================
-- STEP 4: Period comparison — GSC aggregate metrics
-- ====================================================================
-- Compare April MTD vs March same-day-count vs YoY

CREATE OR REPLACE TEMP VIEW gsc_period_comparison AS
WITH periods AS (
  SELECT * FROM _config
)
SELECT
  'April 2026 MTD' AS period,
  COUNT(DISTINCT date)  AS days,
  SUM(clicks)           AS total_clicks,
  SUM(impressions)      AS total_impressions,
  ROUND(SUM(clicks) / NULLIF(COUNT(DISTINCT date), 0), 0) AS avg_daily_clicks,
  ROUND(SUM(impressions) / NULLIF(COUNT(DISTINCT date), 0), 0) AS avg_daily_impressions,
  ROUND(SUM(clicks) * 100.0 / NULLIF(SUM(impressions), 0), 3) AS ctr_pct,
  ROUND(SUM(position * impressions) / NULLIF(SUM(impressions), 0), 1) AS avg_position
FROM lakehouse_production.common.gsc_search_analytics_d_1, periods p
WHERE date BETWEEN p.curr_start AND p.curr_end
  AND domain IN ('choosetexaspower.org','saveonenergy.com','chooseenergy.com','texaselectricrates.com')

UNION ALL

SELECT
  'March 2026 (same days)' AS period,
  COUNT(DISTINCT date),
  SUM(clicks),
  SUM(impressions),
  ROUND(SUM(clicks) / NULLIF(COUNT(DISTINCT date), 0), 0),
  ROUND(SUM(impressions) / NULLIF(COUNT(DISTINCT date), 0), 0),
  ROUND(SUM(clicks) * 100.0 / NULLIF(SUM(impressions), 0), 3),
  ROUND(SUM(position * impressions) / NULLIF(SUM(impressions), 0), 1)
FROM lakehouse_production.common.gsc_search_analytics_d_1, periods p
WHERE date BETWEEN p.prior_start AND p.prior_end
  AND domain IN ('choosetexaspower.org','saveonenergy.com','chooseenergy.com','texaselectricrates.com')

UNION ALL

SELECT
  'April 2025 (YoY)' AS period,
  COUNT(DISTINCT date),
  SUM(clicks),
  SUM(impressions),
  ROUND(SUM(clicks) / NULLIF(COUNT(DISTINCT date), 0), 0),
  ROUND(SUM(impressions) / NULLIF(COUNT(DISTINCT date), 0), 0),
  ROUND(SUM(clicks) * 100.0 / NULLIF(SUM(impressions), 0), 3),
  ROUND(SUM(position * impressions) / NULLIF(SUM(impressions), 0), 1)
FROM lakehouse_production.common.gsc_search_analytics_d_1, periods p
WHERE date BETWEEN p.yoy_start AND p.yoy_end
  AND domain IN ('choosetexaspower.org','saveonenergy.com','chooseenergy.com','texaselectricrates.com');

SELECT * FROM gsc_period_comparison;

## Chart 1: Daily Sessions + GSC Clicks Trend

Overlay current month vs prior month to see the daily shape.

In [ ]:
%python
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

sessions_df = spark.sql("SELECT * FROM sessions_daily").toPandas()
gsc_df = spark.sql("SELECT * FROM gsc_daily").toPandas()

sessions_df['_date'] = pd.to_datetime(sessions_df['_date'])
gsc_df['date'] = pd.to_datetime(gsc_df['date'])

# Split into March and April
s_mar = sessions_df[(sessions_df['_date'] >= '2026-03-01') & (sessions_df['_date'] <= '2026-03-31')].copy()
s_apr = sessions_df[(sessions_df['_date'] >= '2026-04-01')].copy()
g_mar = gsc_df[(gsc_df['date'] >= '2026-03-01') & (gsc_df['date'] <= '2026-03-31')].copy()
g_apr = gsc_df[(gsc_df['date'] >= '2026-04-01')].copy()

# Day-of-month index for overlay
s_mar['dom'] = s_mar['_date'].dt.day
s_apr['dom'] = s_apr['_date'].dt.day
g_mar['dom'] = g_mar['date'].dt.day
g_apr['dom'] = g_apr['date'].dt.day

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Organic Sessions by Day of Month', 'GSC Clicks by Day of Month'),
    vertical_spacing=0.12
)

fig.add_trace(go.Scatter(x=s_mar['dom'], y=s_mar['sessions'], name='March 2026',
    mode='lines+markers', line=dict(color='#636EFA', dash='dash'), marker=dict(size=5)), row=1, col=1)
fig.add_trace(go.Scatter(x=s_apr['dom'], y=s_apr['sessions'], name='April 2026',
    mode='lines+markers', line=dict(color='#EF553B'), marker=dict(size=5)), row=1, col=1)

fig.add_trace(go.Scatter(x=g_mar['dom'], y=g_mar['clicks'], name='March 2026 (GSC)',
    mode='lines+markers', line=dict(color='#636EFA', dash='dash'), marker=dict(size=5),
    showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=g_apr['dom'], y=g_apr['clicks'], name='April 2026 (GSC)',
    mode='lines+markers', line=dict(color='#EF553B'), marker=dict(size=5),
    showlegend=False), row=2, col=1)

fig.update_layout(height=600, title_text='Organic Traffic: April vs March 2026',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1))
fig.update_xaxes(title_text='Day of Month', row=2, col=1)
fig.update_yaxes(title_text='Sessions', row=1, col=1)
fig.update_yaxes(title_text='GSC Clicks', row=2, col=1)
fig.show()

## Chart 2: Impressions and CTR Trend

Separating impressions from CTR reveals whether the decline is a visibility (ranking) problem or a click-through problem.

In [ ]:
%python
fig2 = make_subplots(
    rows=3, cols=1,
    subplot_titles=('GSC Impressions', 'GSC CTR (%)', 'Avg Position (lower=better)'),
    vertical_spacing=0.08
)

for df, name, dash in [(g_mar, 'March', 'dash'), (g_apr, 'April', None)]:
    color = '#636EFA' if name == 'March' else '#EF553B'
    show = (name == 'March')  # only show legend once per series
    fig2.add_trace(go.Scatter(x=df['dom'], y=df['impressions'], name=f'{name} 2026',
        mode='lines+markers', line=dict(color=color, dash=dash), marker=dict(size=4),
        showlegend=show), row=1, col=1)
    fig2.add_trace(go.Scatter(x=df['dom'], y=df['ctr_pct'], name=f'{name} CTR',
        mode='lines+markers', line=dict(color=color, dash=dash), marker=dict(size=4),
        showlegend=False), row=2, col=1)
    fig2.add_trace(go.Scatter(x=df['dom'], y=df['avg_position'], name=f'{name} Position',
        mode='lines+markers', line=dict(color=color, dash=dash), marker=dict(size=4),
        showlegend=False), row=3, col=1)

fig2.update_layout(height=750, title_text='GSC Impressions, CTR & Position: April vs March 2026',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1))
fig2.update_xaxes(title_text='Day of Month', row=3, col=1)
fig2.update_yaxes(title_text='Impressions', row=1, col=1)
fig2.update_yaxes(title_text='CTR %', row=2, col=1)
fig2.update_yaxes(title_text='Avg Position', autorange='reversed', row=3, col=1)
fig2.show()

## Chart 3: Impression × CTR Decomposition (Waterfall)

Decomposes the change in clicks into three components:
- **Impression Effect:** how much of the click change is due to fewer/more impressions (holding CTR constant)
- **CTR Effect:** how much is due to CTR changing (holding impressions constant)
- **Interaction:** the joint effect of both changing simultaneously

Uses DOW-weighted daily averages for a fair partial-month comparison.

In [ ]:
%python
# Period comparison data
comp_df = spark.sql("SELECT * FROM gsc_period_comparison").toPandas()

apr = comp_df[comp_df['period'] == 'April 2026 MTD'].iloc[0]
mar = comp_df[comp_df['period'] == 'March 2026 (same days)'].iloc[0]

# Use daily averages for fair comparison
apr_impr = float(apr['avg_daily_impressions'])
mar_impr = float(mar['avg_daily_impressions'])
apr_ctr  = float(apr['ctr_pct']) / 100
mar_ctr  = float(mar['ctr_pct']) / 100
apr_clicks = float(apr['avg_daily_clicks'])
mar_clicks = float(mar['avg_daily_clicks'])

# Decomposition: ΔClicks = ΔImpr × CTR_prior + Impr_prior × ΔCTR + ΔImpr × ΔCTR
delta_impr = apr_impr - mar_impr
delta_ctr  = apr_ctr - mar_ctr
delta_clicks = apr_clicks - mar_clicks

impression_effect = delta_impr * mar_ctr
ctr_effect        = mar_impr * delta_ctr
interaction       = delta_impr * delta_ctr

print(f"March avg daily: {mar_clicks:.0f} clicks = {mar_impr:,.0f} impr × {mar_ctr*100:.3f}% CTR")
print(f"April avg daily: {apr_clicks:.0f} clicks = {apr_impr:,.0f} impr × {apr_ctr*100:.3f}% CTR")
print(f"")
print(f"Change in daily clicks:  {delta_clicks:+.0f} ({delta_clicks/mar_clicks*100:+.1f}%)")
print(f"  Impression effect:     {impression_effect:+.0f} (impressions {'down' if delta_impr < 0 else 'up'} {abs(delta_impr)/mar_impr*100:.1f}%)")
print(f"  CTR effect:            {ctr_effect:+.0f} (CTR {'down' if delta_ctr < 0 else 'up'} {abs(delta_ctr)/mar_ctr*100:.1f}%)")
print(f"  Interaction:           {interaction:+.0f}")

# Waterfall chart
fig3 = go.Figure(go.Waterfall(
    orientation='v',
    measure=['absolute', 'relative', 'relative', 'relative', 'total'],
    x=['March Avg Daily Clicks', 'Impression Effect', 'CTR Effect', 'Interaction', 'April Avg Daily Clicks'],
    y=[mar_clicks, impression_effect, ctr_effect, interaction, 0],
    text=[f"{mar_clicks:.0f}", f"{impression_effect:+.0f}", f"{ctr_effect:+.0f}", f"{interaction:+.0f}", f"{apr_clicks:.0f}"],
    textposition='outside',
    connector=dict(line=dict(color='rgb(63, 63, 63)')),
    increasing=dict(marker=dict(color='#2CA02C')),
    decreasing=dict(marker=dict(color='#D62728')),
    totals=dict(marker=dict(color='#FF7F0E')),
))
fig3.update_layout(
    title='Click Change Decomposition: April vs March (Avg Daily)',
    yaxis_title='Avg Daily Clicks',
    height=450,
    showlegend=False
)
fig3.show()

## Chart 4: Which SEO Buckets Are Driving the Decline?

Compares April MTD vs March (same days) by SEO query bucket to isolate where impressions and clicks are changing.

In [ ]:
%sql
-- ====================================================================
-- STEP 5: Bucket-level comparison (April vs March, same day count)
-- ====================================================================
-- Uses the rule-based bucket map for consistency.

CREATE OR REPLACE TEMP VIEW bucket_period_comparison AS

WITH bucket_map AS (
  SELECT DISTINCT LOWER(TRIM(query)) AS query,
    CASE
      WHEN LOWER(TRIM(query)) RLIKE '(?i)\\b(luz|electricidad|electrica|energi(a|á)|compa(ñ|n)ia de|servicio de|pagar|cuenta de luz|tarifa|proveedor|sin dep(o|ó)sito|cr(e|é)dito|factura|recibo|planes? de|barato|barata|baratos|baratas|mejor(es)?\\s+(precio|tarifa|plan)|cambiar de|prepago)\\b' THEN 'Spanish'
      WHEN LOWER(TRIM(query)) RLIKE '(?i)(choosetexaspower|choose\\s*texas\\s*power|ctxp|saveonenergy|save\\s*on\\s*energy|chooseenergy|choose\\s*energy|texaselectricrates|texas\\s*electric\\s*rates)' THEN 'Brand'
      WHEN LOWER(TRIM(query)) RLIKE '(?i)\\b(txu|reliant|direct\\s*energ|green\\s*mountain|gexa|champion\\s*energy|cirro|ambit|frontier\\s*(utilit|energy)|pulse\\s*power|4\\s*change|dynowatt|express\\s*energy|discount\\s*power|pennywise|tara\\s*energy|just\\s*energy|mp2|tri\\s*eagle|constellation|shell\\s*energy|octopus\\s*energy|chariot\\s*energy|rhythm|payless\\s*power|first\\s*choice\\s*power|startex|endurance|tomorrow\\s*energy|griddy|spark\\s*energy|veteran\\s*energy|infuse\\s*energy|value\\s*power|our\\s*energy|think\\s*energy|summer\\s*energy|breeze\\s*energy|nrg\\b|centerpoint|oncor|aep\\s*texas|entrust|trieagle|encompass|flagship\\s*power|liberty\\s*power|peco|wtu|bulb\\s*energy|ovo\\s*energy|lumo|eversource|national\\s*grid|nationalgrid|duke\\s*energy|dominion|xcel|alliant|dte\\b|comed\\b|ameren|pepco|bgee?\\b|ppl\\s*electric|firstenergy|exelon|vistra|nrg\\s*energy)\\b' THEN 'Supplier'
      WHEN LOWER(TRIM(query)) RLIKE '(?i)(power\\s*to\\s*choose|powertochoose|power2choose|ptc\\s*texas|puct|powerchoose|apples\\s*to\\s*apples)' THEN 'Aggregator'
      WHEN LOWER(TRIM(query)) RLIKE '(?i)(no deposit|no credit check|prepaid electric|prepaid energy|prepaid light|bad credit|without deposit|no\\s+credito)' THEN 'No Deposit'
      WHEN LOWER(TRIM(query)) RLIKE '(?i)\\b(cheap(est)?|lowest|affordable|low cost|budget|inexpensive|best price|lowest price|most affordable|free electricity|free energy|free nights|free weekends)\\b' THEN 'Price Sensitive'
      WHEN LOWER(TRIM(query)) RLIKE '(?i)\\b(rates?|pricing|price per kwh|cost of (electricity|energy)|kwh (rate|cost|price)|electricity cost|energy cost|how much.*(electricity|energy|kwh|cost)|price to compare|average.*(bill|cost).*electri)\\b' THEN 'Rates'
      WHEN LOWER(TRIM(query)) RLIKE '(?i)(electric(ity|al)?\\s*compan|light\\s*compan|energy\\s*compan|energy\\s*service|electric(ity|al)?\\s*provider|energy\\s*provider|power\\s*compan|electric(ity|al)?\\s*service|energy\\s*supplier|utilit(y|ies)\\s*(compan|provider)|engery\\s*compan|eletric\\s*compan|electrician\\s*compan)' THEN 'Companies'
      WHEN LOWER(TRIM(query)) RLIKE '(?i)\\b(texas|houston|dallas|fort worth|san antonio|austin|corpus christi|el paso|arlington|plano|lubbock|laredo|irving|mcallen|amarillo|brownsville|killeen|midland|odessa|waco|abilene|beaumont|round rock|garland|carrollton|denton|richardson|tyler|college station|wichita falls|galveston|katy|frisco|spring|conroe|sugar land|temple|longview|pearland|pflugerville|allen|edinburg|league city|mission|bryan|pasadena|mesquite|mckinney|san marcos|new braunfels|cedar park|georgetown|leander|euless|duncanville|copperas cove|weatherford|harlingen|victoria|lufkin|nacogdoches|texarkana|del rio|eagle pass|hurst|bedford|mansfield|rowlett|desoto|cedar hill|cleburne|corsicana|rockwall|waxahachie|seguin|bastrop|kyle|buda|liberty hill)\\b' THEN 'Geo'
      WHEN LOWER(TRIM(query)) RLIKE '(?i)\\b(electricity|electric(al)?|energy|power|utility|utilities|light bill|kilowatt|kwh|watt|deregulat|switch.*(provider|plan|service)|energy plan|electricity plan|power plan|renewable|solar|wind energy|green energy|smart meter|grid|thermostat|generator|outage|blackout|brownout)\\b' THEN 'Generic'
      ELSE 'Other'
    END AS query_bucket
  FROM lakehouse_production.common.gsc_search_analytics_d_1
  WHERE date >= '2026-03-01'
    AND domain IN ('choosetexaspower.org','saveonenergy.com','chooseenergy.com','texaselectricrates.com')
    AND query IS NOT NULL AND TRIM(query) != ''
),

tagged AS (
  SELECT
    g.date,
    COALESCE(bm.query_bucket, 'Other') AS query_bucket,
    g.clicks,
    g.impressions,
    g.position
  FROM lakehouse_production.common.gsc_search_analytics_d_1 g
  LEFT JOIN bucket_map bm ON LOWER(TRIM(g.query)) = bm.query
  WHERE g.date >= '2026-03-01'
    AND g.domain IN ('choosetexaspower.org','saveonenergy.com','chooseenergy.com','texaselectricrates.com')
)

SELECT
  CASE
    WHEN date BETWEEN (SELECT curr_start FROM _config) AND (SELECT curr_end FROM _config) THEN 'April'
    WHEN date BETWEEN (SELECT prior_start FROM _config) AND (SELECT prior_end FROM _config) THEN 'March'
  END AS period,
  query_bucket,
  SUM(clicks)      AS clicks,
  SUM(impressions)  AS impressions,
  ROUND(SUM(clicks) * 100.0 / NULLIF(SUM(impressions), 0), 3) AS ctr_pct,
  ROUND(SUM(position * impressions) / NULLIF(SUM(impressions), 0), 1) AS avg_position
FROM tagged
WHERE date BETWEEN (SELECT curr_start FROM _config) AND (SELECT curr_end FROM _config)
   OR date BETWEEN (SELECT prior_start FROM _config) AND (SELECT prior_end FROM _config)
GROUP BY 1, query_bucket
ORDER BY query_bucket, period;

In [ ]:
%python
bucket_df = spark.sql("SELECT * FROM bucket_period_comparison").toPandas()

apr_b = bucket_df[bucket_df['period'] == 'April'].set_index('query_bucket')
mar_b = bucket_df[bucket_df['period'] == 'March'].set_index('query_bucket')

# Merge for comparison
comp = mar_b[['clicks','impressions','ctr_pct','avg_position']].rename(
    columns=lambda c: f'mar_{c}'
).join(
    apr_b[['clicks','impressions','ctr_pct','avg_position']].rename(
        columns=lambda c: f'apr_{c}'
    ),
    how='outer'
).fillna(0)

comp['click_delta'] = comp['apr_clicks'] - comp['mar_clicks']
comp['impr_delta'] = comp['apr_impressions'] - comp['mar_impressions']
comp['click_delta_pct'] = comp['click_delta'] / comp['mar_clicks'].replace(0, float('nan')) * 100
comp['impr_delta_pct'] = comp['impr_delta'] / comp['mar_impressions'].replace(0, float('nan')) * 100
comp = comp.sort_values('click_delta')

# Horizontal bar chart: click change by bucket
fig4 = go.Figure()
fig4.add_trace(go.Bar(
    y=comp.index,
    x=comp['click_delta'],
    orientation='h',
    marker_color=['#D62728' if v < 0 else '#2CA02C' for v in comp['click_delta']],
    text=[f"{v:+,.0f} ({p:+.1f}%)" for v, p in zip(comp['click_delta'], comp['click_delta_pct'])],
    textposition='outside'
))
fig4.update_layout(
    title='GSC Click Change by SEO Bucket (April vs March, same 10 days)',
    xaxis_title='Click Change (absolute)',
    height=450,
    margin=dict(l=120)
)
fig4.show()

# Impression change by bucket
comp_i = comp.sort_values('impr_delta')
fig4b = go.Figure()
fig4b.add_trace(go.Bar(
    y=comp_i.index,
    x=comp_i['impr_delta'],
    orientation='h',
    marker_color=['#D62728' if v < 0 else '#2CA02C' for v in comp_i['impr_delta']],
    text=[f"{v:+,.0f} ({p:+.1f}%)" for v, p in zip(comp_i['impr_delta'], comp_i['impr_delta_pct'])],
    textposition='outside'
))
fig4b.update_layout(
    title='GSC Impression Change by SEO Bucket (April vs March, same 10 days)',
    xaxis_title='Impression Change (absolute)',
    height=450,
    margin=dict(l=120)
)
fig4b.show()

In [ ]:
%python
# Position change by bucket — bubble chart (size = click volume)
comp['position_delta'] = comp['apr_avg_position'] - comp['mar_avg_position']

fig4c = go.Figure()
fig4c.add_trace(go.Scatter(
    x=comp['impr_delta_pct'],
    y=comp['position_delta'],
    mode='markers+text',
    marker=dict(
        size=comp['apr_clicks'].clip(lower=50) / comp['apr_clicks'].max() * 60 + 10,
        color=comp['click_delta_pct'],
        colorscale='RdYlGn',
        colorbar=dict(title='Click Δ%'),
        line=dict(width=1, color='DarkSlateGrey')
    ),
    text=comp.index,
    textposition='top center',
    textfont=dict(size=10)
))
fig4c.update_layout(
    title='Bucket Health Map: Position Change vs Impression Change',
    xaxis_title='Impression Change %',
    yaxis_title='Avg Position Change (positive = worse)',
    height=500,
    yaxis=dict(autorange='reversed'),
    shapes=[dict(type='line', x0=0, x1=0, y0=-5, y1=5, line=dict(dash='dash', color='grey')),
            dict(type='line', x0=-100, x1=100, y0=0, y1=0, line=dict(dash='dash', color='grey'))]
)
fig4c.show()

## Chart 5: Top Pages That Lost the Most Clicks

Identifies the specific landing pages that contributed most to the click decline.

In [ ]:
%sql
-- ====================================================================
-- STEP 6: Page-level click change (top losers and gainers)
-- ====================================================================

CREATE OR REPLACE TEMP VIEW page_click_change AS

WITH apr_pages AS (
  SELECT
    RTRIM('/',
      LOWER(
        CASE WHEN POSITION('#' IN page) > 0
          THEN LEFT(page, POSITION('#' IN page) - 1)
          ELSE page
        END
      )
    ) AS landing_page,
    SUM(clicks)      AS apr_clicks,
    SUM(impressions)  AS apr_impressions,
    ROUND(SUM(clicks) * 100.0 / NULLIF(SUM(impressions), 0), 3) AS apr_ctr_pct,
    ROUND(SUM(position * impressions) / NULLIF(SUM(impressions), 0), 1) AS apr_avg_position
  FROM lakehouse_production.common.gsc_search_analytics_d_1, _config c
  WHERE date BETWEEN c.curr_start AND c.curr_end
    AND domain IN ('choosetexaspower.org','saveonenergy.com','chooseenergy.com','texaselectricrates.com')
  GROUP BY 1
),

mar_pages AS (
  SELECT
    RTRIM('/',
      LOWER(
        CASE WHEN POSITION('#' IN page) > 0
          THEN LEFT(page, POSITION('#' IN page) - 1)
          ELSE page
        END
      )
    ) AS landing_page,
    SUM(clicks)      AS mar_clicks,
    SUM(impressions)  AS mar_impressions,
    ROUND(SUM(clicks) * 100.0 / NULLIF(SUM(impressions), 0), 3) AS mar_ctr_pct,
    ROUND(SUM(position * impressions) / NULLIF(SUM(impressions), 0), 1) AS mar_avg_position
  FROM lakehouse_production.common.gsc_search_analytics_d_1, _config c
  WHERE date BETWEEN c.prior_start AND c.prior_end
    AND domain IN ('choosetexaspower.org','saveonenergy.com','chooseenergy.com','texaselectricrates.com')
  GROUP BY 1
)

SELECT
  COALESCE(a.landing_page, m.landing_page)   AS landing_page,
  COALESCE(m.mar_clicks, 0)                   AS mar_clicks,
  COALESCE(a.apr_clicks, 0)                   AS apr_clicks,
  COALESCE(a.apr_clicks, 0) - COALESCE(m.mar_clicks, 0) AS click_delta,
  COALESCE(m.mar_impressions, 0)              AS mar_impressions,
  COALESCE(a.apr_impressions, 0)              AS apr_impressions,
  COALESCE(a.apr_impressions, 0) - COALESCE(m.mar_impressions, 0) AS impr_delta,
  m.mar_ctr_pct,
  a.apr_ctr_pct,
  m.mar_avg_position,
  a.apr_avg_position,
  COALESCE(a.apr_avg_position, 0) - COALESCE(m.mar_avg_position, 0) AS position_delta
FROM apr_pages a
FULL OUTER JOIN mar_pages m ON a.landing_page = m.landing_page
ORDER BY click_delta ASC;

In [ ]:
%python
page_df = spark.sql("SELECT * FROM page_click_change").toPandas()

# Shorten page URLs for readability
page_df['short_page'] = page_df['landing_page'].str.replace(
    r'https://www\.(choosetexaspower\.org|saveonenergy\.com|chooseenergy\.com|texaselectricrates\.com)',
    lambda m: m.group(1).split('.')[0],
    regex=True
)

# Top 15 losers + top 5 gainers
losers = page_df.nsmallest(15, 'click_delta')
gainers = page_df.nlargest(5, 'click_delta')
display_df = pd.concat([losers, gainers]).drop_duplicates()

fig5 = go.Figure()
fig5.add_trace(go.Bar(
    y=display_df['short_page'],
    x=display_df['click_delta'],
    orientation='h',
    marker_color=['#D62728' if v < 0 else '#2CA02C' for v in display_df['click_delta']],
    text=[f"{v:+,.0f}" for v in display_df['click_delta']],
    textposition='outside'
))
fig5.update_layout(
    title='Top Pages: Click Change (April vs March, same 10 days)',
    xaxis_title='Click Change',
    height=600,
    margin=dict(l=350),
    yaxis=dict(autorange='reversed')
)
fig5.show()

# Print table of biggest losers with position context
print("\nTop 15 pages that lost the most clicks:")
print(losers[['short_page','mar_clicks','apr_clicks','click_delta',
              'mar_avg_position','apr_avg_position','position_delta',
              'mar_ctr_pct','apr_ctr_pct']].to_string(index=False))

## Chart 6: Position Movement Analysis

For pages with meaningful traffic, shows which ones dropped in ranking the most.

In [ ]:
%python
# Filter to pages with at least 5 clicks in either period (meaningful traffic)
meaningful = page_df[
    (page_df['mar_clicks'] >= 5) | (page_df['apr_clicks'] >= 5)
].copy()

meaningful['position_delta'] = meaningful['position_delta'].astype(float)

# Pages that dropped most in position
dropped = meaningful.nlargest(20, 'position_delta')  # positive = worse position
improved = meaningful.nsmallest(10, 'position_delta')  # negative = better position

fig6 = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Biggest Position Drops (worse)', 'Biggest Position Gains (better)'),
    horizontal_spacing=0.25
)

fig6.add_trace(go.Bar(
    y=dropped['short_page'],
    x=dropped['position_delta'],
    orientation='h',
    marker_color='#D62728',
    text=[f"{v:+.1f}" for v in dropped['position_delta']],
    textposition='outside'
), row=1, col=1)

fig6.add_trace(go.Bar(
    y=improved['short_page'],
    x=improved['position_delta'],
    orientation='h',
    marker_color='#2CA02C',
    text=[f"{v:+.1f}" for v in improved['position_delta']],
    textposition='outside'
), row=1, col=2)

fig6.update_layout(
    title='Position Changes: April vs March (pages with ≥5 clicks)',
    height=600,
    showlegend=False,
    margin=dict(l=350)
)
fig6.show()

## Chart 7: DOW-Weighted Pacing Summary

Projects April full-month organic sessions and GSC clicks using day-of-week weights, and compares to March actual and plan.

In [ ]:
%sql
-- ====================================================================
-- STEP 7: DOW-weighted pacing projection for April
-- ====================================================================

CREATE OR REPLACE TEMP VIEW april_pacing AS

WITH dow_weights AS (
  SELECT DAYOFWEEK(_date) AS dow, AVG(daily_sessions) AS avg_daily
  FROM (
    SELECT _date, SUM(sessions) AS daily_sessions
    FROM energy_prod.data_science.mp_session_level_query
    WHERE _date >= '2026-02-01' AND _date < '2026-04-01'
      AND marketing_channel = 'Organic'
    GROUP BY _date
  )
  GROUP BY DAYOFWEEK(_date)
),

april_days AS (
  SELECT
    _date,
    DAYOFWEEK(_date) AS dow,
    CASE WHEN _date <= (SELECT MAX(_date) FROM sessions_daily WHERE _date >= '2026-04-01') THEN 1 ELSE 0 END AS has_data
  FROM (SELECT EXPLODE(SEQUENCE(DATE '2026-04-01', DATE '2026-04-30')) AS _date)
),

actuals AS (
  SELECT SUM(sessions) AS april_mtd_sessions
  FROM sessions_daily
  WHERE _date >= '2026-04-01'
),

march_actual AS (
  SELECT SUM(sessions) AS march_sessions
  FROM sessions_daily
  WHERE _date >= '2026-03-01' AND _date <= '2026-03-31'
)

SELECT
  act.april_mtd_sessions,
  ROUND(
    act.april_mtd_sessions
    / (
      SUM(CASE WHEN ad.has_data = 1 THEN w.avg_daily END)
      / SUM(w.avg_daily)
    )
  , 0) AS april_dow_paced,
  mar.march_sessions,
  48020 AS april_plan,
  ROUND(
    (
      act.april_mtd_sessions
      / (SUM(CASE WHEN ad.has_data = 1 THEN w.avg_daily END) / SUM(w.avg_daily))
      - mar.march_sessions
    ) * 100.0 / mar.march_sessions
  , 1) AS paced_vs_march_pct,
  ROUND(
    (
      act.april_mtd_sessions
      / (SUM(CASE WHEN ad.has_data = 1 THEN w.avg_daily END) / SUM(w.avg_daily))
      - 48020
    ) * 100.0 / 48020
  , 1) AS paced_vs_plan_pct
FROM april_days ad
JOIN dow_weights w ON ad.dow = w.dow
CROSS JOIN actuals act
CROSS JOIN march_actual mar
GROUP BY act.april_mtd_sessions, mar.march_sessions;

SELECT * FROM april_pacing;

In [ ]:
%python
pacing = spark.sql("SELECT * FROM april_pacing").toPandas().iloc[0]

fig7 = go.Figure()
labels = ['March Actual', 'April Plan', 'April DOW Paced', 'April MTD']
values = [
    float(pacing['march_sessions']),
    float(pacing['april_plan']),
    float(pacing['april_dow_paced']),
    float(pacing['april_mtd_sessions'])
]
colors = ['#636EFA', '#AB63FA', '#EF553B', '#FFA15A']

fig7.add_trace(go.Bar(
    x=labels, y=values,
    marker_color=colors,
    text=[f"{v:,.0f}" for v in values],
    textposition='outside'
))
fig7.update_layout(
    title='April 2026 Organic Session Pacing',
    yaxis_title='Sessions',
    height=400
)
fig7.show()

print(f"April DOW-paced projection: {float(pacing['april_dow_paced']):,.0f}")
print(f"  vs March actual: {float(pacing['paced_vs_march_pct']):+.1f}%")
print(f"  vs April plan:   {float(pacing['paced_vs_plan_pct']):+.1f}%")

---
## Chart 8: Top-of-Funnel Visibility (Monthly Trend)

Management view: monthly impressions, clicks, CTR, and weighted average rank in a single 4-panel chart.

- **Weighted avg rank** = impression-weighted position (lower = better)
- April 2026 is paced to full month using `(actual / days_elapsed) × days_in_month`
- Paced value shown as a hollow marker to distinguish from complete months

In [ ]:
%python
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

monthly = spark.sql("""
  SELECT
    DATE_TRUNC('month', date)                                           AS month,
    COUNT(DISTINCT date)                                                AS days_with_data,
    DAY(LAST_DAY(DATE_TRUNC('month', date)))                           AS days_in_month,
    SUM(clicks)                                                         AS clicks,
    SUM(impressions)                                                    AS impressions,
    ROUND(SUM(clicks) * 100.0 / NULLIF(SUM(impressions), 0), 3)       AS ctr_pct,
    ROUND(SUM(position * impressions) / NULLIF(SUM(impressions), 0), 1) AS weighted_avg_rank
  FROM lakehouse_production.common.gsc_search_analytics_d_1
  WHERE domain IN (
    'choosetexaspower.org','saveonenergy.com',
    'chooseenergy.com','texaselectricrates.com'
  )
    AND date >= '2024-01-01'
  GROUP BY DATE_TRUNC('month', date)
  ORDER BY month
""").toPandas()

monthly['month'] = pd.to_datetime(monthly['month']).dt.tz_localize(None)
monthly['days_with_data'] = monthly['days_with_data'].astype(int)
monthly['days_in_month'] = monthly['days_in_month'].astype(int)
monthly['is_partial'] = monthly['days_with_data'] < monthly['days_in_month']

# Pace partial months to full-month equivalents
monthly['clicks_paced'] = monthly.apply(
    lambda r: r['clicks'] * r['days_in_month'] / r['days_with_data'] if r['is_partial'] else r['clicks'], axis=1
)
monthly['impressions_paced'] = monthly.apply(
    lambda r: r['impressions'] * r['days_in_month'] / r['days_with_data'] if r['is_partial'] else r['impressions'], axis=1
)

complete = monthly[~monthly['is_partial']]
partial  = monthly[monthly['is_partial']]

fig8 = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Monthly Impressions', 'Monthly Clicks',
        'CTR (%)', 'Weighted Avg Rank (lower = better)'
    ),
    vertical_spacing=0.14, horizontal_spacing=0.10
)

line_style = dict(color='#636EFA', width=2.5)
marker_style = dict(size=6, color='#636EFA')
paced_marker = dict(size=10, color='white', line=dict(color='#EF553B', width=2.5), symbol='circle-open')

# Impressions
fig8.add_trace(go.Scatter(
    x=complete['month'], y=complete['impressions'], mode='lines+markers',
    name='Impressions', line=line_style, marker=marker_style, showlegend=False
), row=1, col=1)
if not partial.empty:
    fig8.add_trace(go.Scatter(
        x=partial['month'], y=partial['impressions_paced'], mode='markers',
        name='Paced', marker=paced_marker, showlegend=True
    ), row=1, col=1)

# Clicks
fig8.add_trace(go.Scatter(
    x=complete['month'], y=complete['clicks'], mode='lines+markers',
    name='Clicks', line=line_style, marker=marker_style, showlegend=False
), row=1, col=2)
if not partial.empty:
    fig8.add_trace(go.Scatter(
        x=partial['month'], y=partial['clicks_paced'], mode='markers',
        marker=paced_marker, showlegend=False
    ), row=1, col=2)

# CTR
fig8.add_trace(go.Scatter(
    x=monthly['month'], y=monthly['ctr_pct'], mode='lines+markers',
    name='CTR', line=line_style, marker=marker_style, showlegend=False
), row=2, col=1)

# Weighted Avg Rank (inverted: lower is better)
fig8.add_trace(go.Scatter(
    x=monthly['month'], y=monthly['weighted_avg_rank'], mode='lines+markers',
    name='Weighted Avg Rank', line=dict(color='#EF553B', width=2.5),
    marker=dict(size=6, color='#EF553B'), showlegend=False
), row=2, col=2)

fig8.update_yaxes(autorange='reversed', row=2, col=2)
fig8.update_layout(
    height=650,
    title_text='SEO Top-of-Funnel Visibility — Monthly Trend (Jan 2024 – Apr 2026)',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    margin=dict(t=100)
)

for r in range(1, 3):
    for c in range(1, 3):
        fig8.update_xaxes(dtick='M3', tickformat='%b\n%Y', row=r, col=c)

fig8.show()

# Print the last 6 months for quick reference
print("\nLast 6 months (April paced to full month):")
recent = monthly.tail(6).copy()
recent['display_clicks'] = recent.apply(
    lambda r: f"{int(r['clicks_paced']):,}*" if r['is_partial'] else f"{int(r['clicks']):,}", axis=1
)
recent['display_impr'] = recent.apply(
    lambda r: f"{int(r['impressions_paced']):,}*" if r['is_partial'] else f"{int(r['impressions']):,}", axis=1
)
for _, r in recent.iterrows():
    print(f"  {r['month'].strftime('%b %Y'):>8s}:  {r['display_impr']:>14s} impr  |  {r['display_clicks']:>8s} clicks  |  {float(r['ctr_pct']):.3f}% CTR  |  rank {float(r['weighted_avg_rank']):.1f}")

---
## Summary & Diagnostic Framework

### How to read these charts:

| Signal | Primary cause | What to investigate |
|--------|---------------|--------------------|
| Impressions down, CTR flat | **Ranking loss** | Position changes, algorithm update, new competitors |
| Impressions flat, CTR down | **SERP changes** | Featured snippets, AI overviews, competitor snippets |
| Impressions down, CTR up | **Lost low-quality impressions** | Often benign: lost tail queries that never clicked anyway |
| Both down | **Major ranking event** | Check Google algo updates, manual actions, technical issues |

### Next steps depending on findings:
1. **If ranking dropped on specific pages:** Check for technical issues (indexing, canonicals, page speed), content freshness, or competitor gains
2. **If impressions dropped across the board:** Check for Google algorithm updates around the date of decline (Search Engine Roundtable, Search Engine Land)
3. **If concentrated in one bucket:** Competitive landscape may have shifted for that keyword category
4. **If CTR dropped but rankings are stable:** SERP feature changes (AI overviews, featured snippets displacing organic clicks)